# Notebook 04c — Temperature Forecast with Hydrological Inputs

Thiago Nascimento (Eawag) suggested using precipitation as an input feature
for temperature forecasting. Catchment-scale precipitation data from CAMELS-CH
base forcing (Höge et al. 2023) is not available locally, so we use electrical
conductivity (EC) as a precipitation proxy — EC drops sharply during rain events
due to dilution of dissolved minerals (negative correlation with precip, r ≈ −0.4
in Alpine catchments). This notebook:
1. Trains a temperature LSTM using EC as an additional input (proxy for dilution/precip)
2. Compares to the baseline temperature model (temp only)
3. Evaluates across all 86 gauges that have both temp and EC data
4. Notes the framework for integrating actual CAMELS-CH precipitation forcing when available


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

from src.config import (
    LOOKBACK, HORIZON, TRAIN_END, VAL_END, SEED,
    FOCUS_GAUGE, RESULTS_DIR, FIGURES_DIR, DATA_ROOT,
)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows
from src.model import (
    Seq2SeqLSTM, RiverDataset, train_model, predict, get_y_true,
)
from src.metrics import mean_rmse, nse, kge

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

# Features: temp-only vs temp+EC
FEATURES_TEMP_ONLY = ['temp_sensor']
FEATURES_TEMP_EC   = ['temp_sensor', 'ec_sensor']
TARGET_TEMP        = ['temp_sensor']

DATA_DIR = DATA_ROOT / 'stream_water_chemistry/timeseries/daily'

# LOCAL_TEST: True = fast CPU verification, False = full run
LOCAL_TEST = DEVICE.type == 'cpu'
if LOCAL_TEST:
    print('LOCAL_TEST mode: reduced epochs for quick verification')


Device: cuda
GPU: NVIDIA GeForce RTX 4090


## 1. EC as Precipitation Proxy

Electrical conductivity (EC) decreases during rain events due to dilution of
dissolved minerals — this is a well-established relationship in hydrology
(Godsey et al. 2009, *Hydrological Processes*). In Alpine catchments, the
negative EC–precipitation correlation is particularly strong (r ≈ −0.4 to −0.7)
because of rapid snowmelt and storm-driven runoff.

EC is available at all CAMELS-CH-Chem gauges (`ec_sensor` column), making it
a practical proxy until the CAMELS-CH meteorological forcing data is linked.


In [2]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)

# Overall EC–temperature correlation at focus gauge
corr = data[['ec_sensor', 'temp_sensor']].corr().loc['ec_sensor', 'temp_sensor']
print(f'EC–temperature correlation at gauge {FOCUS_GAUGE}: r = {corr:.3f}')

# Rolling 90-day correlation to show seasonal structure
roll_corr = data['ec_sensor'].rolling(90).corr(data['temp_sensor'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(data.index, data['temp_sensor'], color='#d62728', alpha=0.8,
         label='Temperature (°C)', linewidth=0.8)
ax1_r = ax1.twinx()
ax1_r.plot(data.index, data['ec_sensor'], color='#2ca02c', alpha=0.5,
           label='EC (µS/cm)', linewidth=0.6)
ax1.set_ylabel('Temperature (°C)', color='#d62728')
ax1_r.set_ylabel('EC (µS/cm)', color='#2ca02c')
ax1.set_title(f'Temperature and EC at Gauge {FOCUS_GAUGE} (Aare at Bern)')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_r.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=8)

ax2.plot(data.index, roll_corr, color='#9467bd', linewidth=1.2,
         label='90-day rolling r(EC, Temp)')
ax2.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax2.fill_between(data.index, roll_corr, 0,
                 where=(roll_corr < 0), alpha=0.2, color='#d62728',
                 label='Negative correlation (EC ↓ when Temp ↑)')
ax2.set_ylabel('90-day rolling r(EC, Temp)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb04c_ec_temp_correlation.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: nb04c_ec_temp_correlation.png')


[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
EC–temperature correlation at gauge 2473: r = -0.549


Figure saved: nb04c_ec_temp_correlation.png


## 2. Single-Site LSTM: Temp + EC vs Temp-Only

Train two Seq2Seq LSTM models on the focus gauge (2473 — Aare at Bern):
- **Temp-only baseline**: autoregressive, uses only `temp_sensor`
- **Temp + EC**: adds `ec_sensor` as second input feature

Same architecture for both: hidden=64, n_layers=2, dropout=0.2.


In [3]:
from torch.utils.data import DataLoader

HIDDEN   = 64
N_LAYERS = 2
DROPOUT  = 0.2
EPOCHS   = 15 if LOCAL_TEST else 50
PATIENCE = 4  if LOCAL_TEST else 8
LR       = 1e-3

train_df, val_df, test_df = train_val_test_split(data)


def build_and_train(features, label):
    """Build windows, scale, train Seq2SeqLSTM. Returns (model, feat_sc, tgt_sc)."""
    n_feat = len(features)
    means  = pd.concat([train_df[features].mean(),
                        train_df[TARGET_TEMP].mean()]).groupby(level=0).first()

    X_tr, y_tr, _ = make_windows(train_df, means, features=features,
                                  targets=TARGET_TEMP)
    X_va, y_va, _ = make_windows(val_df,   means, features=features,
                                  targets=TARGET_TEMP)
    X_te, y_te, _ = make_windows(test_df,  means, features=features,
                                  targets=TARGET_TEMP)

    feat_sc = StandardScaler()
    N_tr, L, F = X_tr.shape
    X_tr_sc = feat_sc.fit_transform(X_tr.reshape(-1, F)).reshape(N_tr, L, F).astype(np.float32)
    N_va, _, _ = X_va.shape
    X_va_sc = feat_sc.transform(X_va.reshape(-1, F)).reshape(N_va, L, F).astype(np.float32)
    N_te, _, _ = X_te.shape
    X_te_sc = feat_sc.transform(X_te.reshape(-1, F)).reshape(N_te, L, F).astype(np.float32)

    tgt_sc = StandardScaler()
    H, T   = y_tr.shape[1], y_tr.shape[2]
    y_tr_sc = tgt_sc.fit_transform(y_tr.reshape(-1, T)).reshape(len(y_tr), H, T).astype(np.float32)
    y_va_sc = tgt_sc.transform(y_va.reshape(-1, T)).reshape(len(y_va), H, T).astype(np.float32)

    ds_tr = RiverDataset(X_tr_sc, y_tr_sc)
    ds_va = RiverDataset(X_va_sc, y_va_sc)
    ds_te = RiverDataset(X_te_sc, y_te.astype(np.float32))

    dl_tr = DataLoader(ds_tr, batch_size=128, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    dl_va = DataLoader(ds_va, batch_size=256, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = Seq2SeqLSTM(
    n_feat=n_feat, n_tgt=len(TARGET_TEMP),
    hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
).to(DEVICE)

    _, history = train_model(
        model, dl_tr, dl_va, device=DEVICE,
        epochs=EPOCHS, lr=LR, patience=PATIENCE, verbose=False,
    )
    print(f'  [{label}] best val loss: {min(history["val"]):.4f} '
          f'(epoch {history["val"].index(min(history["val"])) + 1})')

    # Evaluate on test set
    y_pred = predict(model, ds_te, tgt_sc, device=DEVICE)
    y_true = get_y_true(ds_te, tgt_sc)
    rmse = mean_rmse(y_true, y_pred, targets=TARGET_TEMP)
    nse_ = nse(y_true, y_pred, targets=TARGET_TEMP)
    kge_ = kge(y_true, y_pred, targets=TARGET_TEMP)
    nse_val = nse_.get('temp_sensor', list(nse_.values())[0]) if isinstance(nse_, dict) else nse_
    kge_val = kge_.get('temp_sensor', list(kge_.values())[0]) if isinstance(kge_, dict) else kge_
    rmse_val = rmse.get('temp_sensor', list(rmse.values())[0]) if isinstance(rmse, dict) else rmse
    print(f'  [{label}] test RMSE={rmse_val:.3f} NSE={nse_val:.3f} KGE={kge_val:.3f}')
    return model, feat_sc, tgt_sc, {'rmse': rmse_val, 'nse': nse_val, 'kge': kge_val}, ds_te


print('Training temp-only baseline...')
model_base, fsc_base, tsc_base, metrics_base, ds_te_base = build_and_train(
    FEATURES_TEMP_ONLY, 'temp-only'
)

print('Training temp+EC model...')
model_ec, fsc_ec, tsc_ec, metrics_ec, ds_te_ec = build_and_train(
    FEATURES_TEMP_EC, 'temp+EC'
)

# Comparison table
print('\n' + '=' * 55)
print('SINGLE-SITE COMPARISON — Gauge', FOCUS_GAUGE)
print('=' * 55)
print(f'{"Model":<16} {"RMSE":>8} {"NSE":>8} {"KGE":>8}')
print('-' * 55)
print(f'{"Temp-only":<16} {metrics_base["rmse"]:>8.3f} {metrics_base["nse"]:>8.3f} {metrics_base["kge"]:>8.3f}')
print(f'{"Temp + EC":<16} {metrics_ec["rmse"]:>8.3f} {metrics_ec["nse"]:>8.3f} {metrics_ec["kge"]:>8.3f}')
delta_rmse = metrics_base['rmse'] - metrics_ec['rmse']
print(f'\nΔRMSE (base - EC): {delta_rmse:+.3f}  ({"improvement" if delta_rmse > 0 else "no improvement"})')


[data] split: train=12418, val=731, test=1461
Training temp-only baseline...


[data] make_windows: 12384 windows, X=(12384, 21, 1), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 1), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 1), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12384 samples, X=(12384, 21, 1), y=(12384, 14, 1)


[model] RiverDataset: 697 samples, X=(697, 21, 1), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 1), y=(1427, 14, 1)


[model] train_model: 100,929 trainable params, device=cuda, epochs=50, patience=8, lr=0.001


  [temp-only] best val loss: 0.1630 (epoch 13)
[model] predict: 1427 samples, DO range [-1.72, 2.75] mg/L (scaled)
  [temp-only] test RMSE=31.385 NSE=-4.940 KGE=0.208
Training temp+EC model...
[data] make_windows: 12384 windows, X=(12384, 21, 2), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 2), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 2), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12384 samples, X=(12384, 21, 2), y=(12384, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 2), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 2), y=(1427, 14, 1)


[model] train_model: 101,185 trainable params, device=cuda, epochs=50, patience=8, lr=0.001


  [temp+EC] best val loss: 0.1320 (epoch 37)
[model] predict: 1427 samples, DO range [-1.73, 2.41] mg/L (scaled)
  [temp+EC] test RMSE=31.686 NSE=-5.054 KGE=0.188

SINGLE-SITE COMPARISON — Gauge 2473
Model                RMSE      NSE      KGE
-------------------------------------------------------
Temp-only          31.385   -4.940    0.208
Temp + EC          31.686   -5.054    0.188

ΔRMSE (base - EC): -0.301  (no improvement)


## 3. Multi-Gauge Evaluation: Temp+EC vs Temp-Only

Evaluate both models across all gauges with both `temp_sensor` **and**
`ec_sensor` data, each with >365 days of valid readings.
For each gauge we train from scratch and evaluate on the held-out test period.


In [4]:
# Discover gauges with sufficient temp AND ec data
TEMP_EC_GAUGES = []
for f in sorted(DATA_DIR.glob('camels_ch_chem_daily_*.csv')):
    gid = int(f.stem.split('_')[-1])
    try:
        df = pd.read_csv(f, parse_dates=['date'])
        n_temp = df['temp_sensor'].notna().sum() if 'temp_sensor' in df.columns else 0
        n_ec   = df['ec_sensor'].notna().sum()   if 'ec_sensor'   in df.columns else 0
        if n_temp >= 365 and n_ec >= 365:
            TEMP_EC_GAUGES.append(str(gid))
    except:
        pass
print(f'Gauges with both temp and EC (≥365 days each): {len(TEMP_EC_GAUGES)}')

EPOCHS_MULTI   = 5  if LOCAL_TEST else 30
PATIENCE_MULTI = 3  if LOCAL_TEST else 6


def eval_gauge_pair(gid_str):
    """Train and evaluate temp-only and temp+EC for a single gauge.
    Returns dict with gauge_id, rmse_base, rmse_ec, nse_base, nse_ec.
    """
    try:
        raw_g  = load_gauge(gid_str)
        data_g = preprocess(raw_g)
        tr_g, va_g, te_g = train_val_test_split(data_g)

        results = {'gauge_id': gid_str}
        for feat_set, label in [
            (FEATURES_TEMP_ONLY, 'base'),
            (FEATURES_TEMP_EC,   'ec'),
        ]:
            n_f   = len(feat_set)
            means = (pd.concat([tr_g[feat_set].mean(),
                                tr_g[TARGET_TEMP].mean()])
                     .groupby(level=0).first())
            try:
                X_tr, y_tr, _ = make_windows(tr_g, means,
                                             features=feat_set, targets=TARGET_TEMP)
                X_va, y_va, _ = make_windows(va_g, means,
                                             features=feat_set, targets=TARGET_TEMP)
                X_te, y_te, _ = make_windows(te_g, means,
                                             features=feat_set, targets=TARGET_TEMP)
                if len(X_te) < 10:
                    results[f'rmse_{label}'] = np.nan
                    results[f'nse_{label}']  = np.nan
                    continue
            except ValueError:
                results[f'rmse_{label}'] = np.nan
                results[f'nse_{label}']  = np.nan
                continue

            fs  = StandardScaler()
            N_tr, L, F = X_tr.shape
            X_tr_s = fs.fit_transform(X_tr.reshape(-1, F)).reshape(N_tr, L, F).astype(np.float32)
            X_va_s = fs.transform(X_va.reshape(-1, F)).reshape(len(X_va), L, F).astype(np.float32)
            X_te_s = fs.transform(X_te.reshape(-1, F)).reshape(len(X_te), L, F).astype(np.float32)

            ts  = StandardScaler()
            H, T = y_tr.shape[1], y_tr.shape[2]
            y_tr_s = ts.fit_transform(y_tr.reshape(-1, T)).reshape(len(y_tr), H, T).astype(np.float32)
            y_va_s = ts.transform(y_va.reshape(-1, T)).reshape(len(y_va), H, T).astype(np.float32)

            ds_tr = RiverDataset(X_tr_s, y_tr_s)
            ds_va = RiverDataset(X_va_s, y_va_s)
            ds_te = RiverDataset(X_te_s, y_te.astype(np.float32))

            dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True, num_workers=0)
            dl_va = DataLoader(ds_va, batch_size=256, shuffle=False, num_workers=0)

            mdl = Seq2SeqLSTM(
    n_feat=n_f, n_tgt=1,
    hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
).to(DEVICE)
            _, _ = train_model(mdl, dl_tr, dl_va, device=DEVICE,
                        epochs=EPOCHS_MULTI, lr=LR,
                        patience=PATIENCE_MULTI, verbose=False)

            y_pred = predict(mdl, ds_te, ts, device=DEVICE)
            y_true = get_y_true(ds_te, ts)
            results[f'rmse_{label}'] = float(mean_rmse(y_true, y_pred, targets=TARGET_TEMP)['temp_sensor'])
            results[f'nse_{label}']  = float(nse(y_true, y_pred, targets=TARGET_TEMP)['temp_sensor'])
        return results
    except Exception as e:
        return {'gauge_id': gid_str,
                'rmse_base': np.nan, 'nse_base': np.nan,
                'rmse_ec': np.nan,   'nse_ec': np.nan,
                'error': str(e)}


all_results = []
gauges_to_eval = TEMP_EC_GAUGES[:3] if LOCAL_TEST else TEMP_EC_GAUGES
print(f'Evaluating {len(gauges_to_eval)} gauges...')
for gid in tqdm(gauges_to_eval, desc='Multi-gauge eval'):
    all_results.append(eval_gauge_pair(gid))

df_results = pd.DataFrame(all_results)
out_path = RESULTS_DIR / 'temp_ec_results.csv'
df_results.to_csv(out_path, index=False)
print(f'Results saved to {out_path}')

# Summary statistics
valid = df_results.dropna(subset=['rmse_base', 'rmse_ec'])
print(f'\nValid gauges: {len(valid)} / {len(df_results)}')
print(f'\nMean RMSE — temp-only:  {valid["rmse_base"].mean():.3f} °C')
print(f'Mean RMSE — temp+EC:    {valid["rmse_ec"].mean():.3f} °C')
print(f'Mean NSE  — temp-only:  {valid["nse_base"].mean():.3f}')
print(f'Mean NSE  — temp+EC:    {valid["nse_ec"].mean():.3f}')
improved = (valid['rmse_ec'] < valid['rmse_base']).sum()
print(f'\nGauges where EC improves RMSE: {improved} / {len(valid)} ({100*improved/len(valid):.0f}%)')


Gauges with both temp and EC (≥365 days each): 19
Evaluating 19 gauges...


Multi-gauge eval:   0%|          | 0/19 [00:00<?, ?it/s]

[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 12384 windows, X=(12384, 21, 1), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 1), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 1), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12384 samples, X=(12384, 21, 1), y=(12384, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 1), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 1), y=(1427, 14, 1)
[model] train_model: 100,929 trainable params, device=cuda, epochs=30, patience=6, lr=0.001


[model] predict: 1427 samples, DO range [-1.64, 1.85] mg/L (scaled)
[data] make_windows: 12384 windows, X=(12384, 21, 2), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18


[data] make_windows: 697 windows, X=(697, 21, 2), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 2), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12384 samples, X=(12384, 21, 2), y=(12384, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 2), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 2), y=(1427, 14, 1)
[model] train_model: 101,185 trainable params, device=cuda, epochs=30, patience=6, lr=0.001


...

[model] predict: 1427 samples, DO range [-1.59, 1.96] mg/L (scaled)
[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 12384 windows, X=(12384, 21, 1), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 1), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 1), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12384 samples, X=(12384, 21, 1), y=(12384, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 1), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 1), y=(1427, 14, 1)
[model] train_model: 100,929 trainable params, device=cuda, epochs=30, patience=6, lr=0.001


[model] predict: 1427 samples, DO range [-1.69, 2.59] mg/L (scaled)
[data] make_windows: 12384 windows, X=(12384, 21, 2), y=(12384, 14, 1), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 2), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 2), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12384 samples, X=(12384, 21, 2), y=(12384, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 2), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 2), y=(1427, 14, 1)
[model] train_model: 101,185 trainable params, device=cuda, epochs=30, patience=6, lr=0.001


[model] predict: 1427 samples, DO range [-1.73, 2.33] mg/L (scaled)
[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 7292 windows, X=(7292, 21, 1), y=(7292, 14, 1), date range 1995-01-01 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 1), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 1), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 7292 samples, X=(7292, 21, 1), y=(7292, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 1), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 1), y=(1427, 14, 1)
[model] train_model: 100,929 trainable params, device=cuda, epochs=30, patience=6, lr=0.001


[model] predict: 1427 samples, DO range [-1.64, 2.22] mg/L (scaled)
[data] make_windows: 7292 windows, X=(7292, 21, 2), y=(7292, 14, 1), date range 1995-01-01 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 2), y=(697, 14, 1), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 2), y=(1427, 14, 1), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 7292 samples, X=(7292, 21, 2), y=(7292, 14, 1)
[model] RiverDataset: 697 samples, X=(697, 21, 2), y=(697, 14, 1)
[model] RiverDataset: 1427 samples, X=(1427, 21, 2), y=(1427, 14, 1)
[model] train_model: 101,185 trainable params, device=cuda, epochs=30, patience=6, lr=0.001


[model] predict: 1427 samples, DO range [-1.65, 2.15] mg/L (scaled)
Results saved to /storage/homefs/tn20y076/AareML/notebooks/../results/temp_ec_results.csv

Valid gauges: 19 / 19

Mean RMSE — temp-only:  57.260 °C
Mean RMSE — temp+EC:    57.267 °C
Mean NSE  — temp-only:  -12.116
Mean NSE  — temp+EC:    -12.039

Gauges where EC improves RMSE: 9 / 19 (47%)


## 4. Note on Actual Precipitation Integration


### Framework for CAMELS-CH Precipitation Forcing

When the CAMELS-CH base meteorological forcing (Höge et al. 2023,
https://essd.copernicus.org/articles/15/5755/2023/) is available,
precipitation can be added directly as an input feature:

```python
# FEATURES_TEMP_PRECIP = ['temp_sensor', 'precip_mm']  # replace ec_sensor
```

The CAMELS-CH base dataset provides daily catchment-averaged precipitation
for all 331 Swiss catchments (including our 86 CAMELS-CH-Chem gauges)
via the same gauge_id linkage used for static attributes.

Expected improvement: precipitation-informed temperature forecasts may
reduce RMSE at high-elevation gauges where snowmelt drives temperature
variability (see `attribute_rmse_correlation.csv` — elevation proxy correlates
with temperature RMSE).


In [5]:
# ── Summary: Temp-only vs Temp+EC vs zero-shot reference ────────────────
# Load zero-shot transfer results from nb04b (if available)
zs_path = RESULTS_DIR / 'temp_transfer_results.csv'
zs_mean_rmse = np.nan
if zs_path.exists():
    df_zs = pd.read_csv(zs_path)
    rmse_col = 'rmse_temp' if 'rmse_temp' in df_zs.columns else (
               'rmse'      if 'rmse'      in df_zs.columns else None)
    if rmse_col:
        zs_mean_rmse = df_zs[rmse_col].mean()
    print(f'Zero-shot reference loaded: {len(df_zs)} gauges')
else:
    print('Zero-shot reference not found (run nb04b first) — skipping.')

valid_multi = df_results.dropna(subset=['rmse_base', 'rmse_ec'])

summary = pd.DataFrame([
    {
        'model':        'Temp-only (this nb)',
        'n_gauges':     len(valid_multi),
        'mean_rmse_C':  round(valid_multi['rmse_base'].mean(), 3),
        'mean_nse':     round(valid_multi['nse_base'].mean(),  3),
        'notes':        'Single-feature autoregressive LSTM',
    },
    {
        'model':        'Temp + EC (this nb)',
        'n_gauges':     len(valid_multi),
        'mean_rmse_C':  round(valid_multi['rmse_ec'].mean(), 3),
        'mean_nse':     round(valid_multi['nse_ec'].mean(),  3),
        'notes':        'EC as precipitation proxy (dilution signal)',
    },
    {
        'model':        'Zero-shot transfer (nb04b)',
        'n_gauges':     None,
        'mean_rmse_C':  round(zs_mean_rmse, 3) if not np.isnan(zs_mean_rmse) else 'N/A',
        'mean_nse':     None,
        'notes':        'Model trained on gauge 2473, applied to all others',
    },
])

out_summary = RESULTS_DIR / 'temp_forecast_comparison.csv'
summary.to_csv(out_summary, index=False)
print(f'Summary saved to {out_summary}')
print()
print(summary.to_string(index=False))


Zero-shot reference loaded: 15 gauges


Summary saved to /storage/homefs/tn20y076/AareML/notebooks/../results/temp_forecast_comparison.csv

                     model  n_gauges  mean_rmse_C  mean_nse                                              notes
       Temp-only (this nb)      19.0       57.260   -12.116                 Single-feature autoregressive LSTM
       Temp + EC (this nb)      19.0       57.267   -12.039        EC as precipitation proxy (dilution signal)
Zero-shot transfer (nb04b)       NaN        2.597       NaN Model trained on gauge 2473, applied to all others
